In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets
%dotenv ../../05_src/.env
import sys
sys.path.append('../../05_src/')

In [2]:
from utils.clients import get_client
client = get_client()

In [3]:
# Import packages
from langchain.chat_models import init_chat_model
import os

In [4]:
# Initialize model
model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0.7,
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

# Tools

In [57]:
import requests
from urllib.parse import quote
from langchain.tools import tool
import json

In [ ]:
@tool
def get_lyrics(title:str, artist: str, timeout: int = 10) -> str | None:
    """
    Retrieve song lyrics from the Lyrics.ovh API.

    Args:
        artist: Artist name.
        title: Song title.
        timeout: Request timeout in seconds.

    Returns:
        The lyrics as a string if found, otherwise None.
    """
    artist = quote(artist.strip())
    title = quote(title.strip())

    url = f"https://api.lyrics.ovh/v1/{artist}/{title}"

    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()

        data = response.json()
        return data.get("lyrics")

    except requests.exceptions.HTTPError:
        print(f"Lyrics not found for '{title}' by '{artist}'.")
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")

    return None
    

In [7]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")
    
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

In [199]:
@tool
def get_artist_history(artist: str) -> str:
    """
    Uses a simple web search to return the history of a musical artist.
    """
    tavily_search_tool = TavilySearch(
        max_results = 5,
        topic = "general"
    )
    results = tavily_search_tool.invoke({"query": f"{artist} music artist history biography"})
    return results

## Semantic Search

In [40]:
from IPython.display import display, Markdown
from tqdm import tqdm
import json
import os

MODEL = os.getenv('MODEL', 'gpt-4o-mini')
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL')
DOC_DIR = "../../05_src/documents/pitchfork/chroma_inputs.jsonl" # contains embeddings and metadata
CHROMA_URL = "../../05_src/assignment_chat"

In [41]:
# load jsonl file from local directory
with open(DOC_DIR, "r", encoding="utf-8") as f:
        chroma_inputs = [json.loads(line) for line in f]

In [42]:
# Persistent Client
import chromadb
chroma_client = chromadb.PersistentClient(path = "../../05_src/assignment_chat")

COLLECTION_NAME = "pitchfork_reviews"

In [43]:
def setup_collection(collection_name: str = COLLECTION_NAME):
    # check if collection exists, if yes, delete that collection
    print(f"Creating collection: {collection_name}")
    collections = chroma_client.list_collections()
    print("Existing:", [c.name for c in collections])

    if collection_name in [col.name for col in collections]:
        print("Deleting existing collection")
        chroma_client.delete_collection(name=collection_name)
    
    # create new collection
    collection = chroma_client.create_collection(
        name=collection_name)
    print("Created:", collection.name)
    
    return collection

In [44]:
def load_embeddings_to_db(chroma_inputs:list[dict],
                          collection_name:str,
                          batch_size:int=1000
                          ):
    # calls function to set-up new collection
    collection = setup_collection(collection_name=collection_name)
    
    # add items from jsonl file to collection using batches to reduce memory usage
    # will show a progress bar
    for i in tqdm(range(0, len(chroma_inputs), batch_size)):
        batch = chroma_inputs[i:i + batch_size]
        # create new list of sequential ids '1', '2'
        ids = [
            f"{j}"
            for j in range(i, i + len(batch))
        ]
        try:
        # add list in bach to collection
            collection.add(
                documents=[item['text'] for item in batch],
                embeddings=[item['embedding'] for item in batch],
                metadatas=[item['metadata'] for item in batch],
                ids=ids
            )
        except Exception as e:
            print(e)
            raise
    return collection

In [46]:
# Create collections vector database in local memory
collection = load_embeddings_to_db(chroma_inputs = chroma_inputs,
                              collection_name=COLLECTION_NAME,
                              batch_size=1000)

Creating collection: pitchfork_reviews
Existing: ['pitchfork_reviews']
Deleting existing collection
Created: pitchfork_reviews


100%|██████████| 20/20 [00:50<00:00,  2.52s/it]


In [47]:
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

In [48]:
def query_chromadb(query, top_n = 2):
    query_embedding = get_embedding(query)
    results = collection.query(query_embeddings = [query_embedding], n_results = top_n)
    return [(id, score, text) for id, score, text in zip(results['ids'][0], results['distances'][0], results['documents'][0])]

In [ ]:
@tool
def semantic_search(query: str) -> str:
    """
    Search the local music database from pitchfork reviews using semantic simalarity.
    Use this tool when the user asks for music or artist suggestions.
    """
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=2,
        include=["documents", "metadatas"]
    )

    docs = results["documents"][0]
    metadata = results["metadatas"][0]

    if not docs:
        return "No matching documents found."

    output = []

    for i, (doc, meta) in enumerate(zip(docs, metadata), start=1):
        output.append(
            f"Result {i}\n"
            f"Metadata: {meta}\n\n"
            f"{doc}"
        )

    return "\n\n---\n\n".join(output)

# Compile Agent

In [249]:
# Augment the LLM with tools
tools = [get_lyrics, get_artist_history, semantic_search]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

Define State

In [253]:
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

Define Model Node

In [254]:
from langchain_core.messages import SystemMessage


def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""
    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content="You are a helpful assistant tasked with returning lyrics to a song given an artist and song title or an artist history given an artist name."
                    )
                ]
                + state["messages"]
            )
        ],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

Define Tool Node

In [255]:
from langchain_core.messages import ToolMessage


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

Define End Logic

In [256]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

Build and Compile the Agent

In [257]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent. Note this is a LangGraph agent, not a classic LangChain AgentExecutor
agent = agent_builder.compile()

# Gradio UI

In [82]:
import gradio as gr
from typing import Optional

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
import ast

# Helper function to output full song lyrics
def extract_tool_result(messages, target_tool):
    """
    Searches the message log to see if target_tool variable is called. It will be "get_lyrics" tool. 
    If found, final message is outputed as tool message for full lyrics instead of final AI message.
    """
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for call in msg.tool_calls:
                if call["name"] == target_tool:
                    tool_call_id = call["id"]

                    for tool_msg in messages:
                        if (
                            isinstance(tool_msg, ToolMessage)
                            and tool_msg.tool_call_id == tool_call_id
                        ):
                            return tool_msg.content

    return None

In [ ]:
def music_chat(message, history):
    """
    Invokes compiled agent and outputs AI message given the user input.
    """
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=message)
            ]
        }
    )

    messages = response["messages"]

    # If lyric_tool is called, output full tool result instead of message passed through AI
    lyrics_result = extract_tool_result(
        messages,
        "get_lyrics"
    )

    # for degbugging only
    #for msg in response["messages"]:
    #    print("\n---")
    #    print(type(msg).__name__)
    #    print(msg)
        
    if lyrics_result:
        return lyrics_result

    return messages[-1].content

In [268]:
demo = gr.ChatInterface(
    fn=music_chat,
    title="Music Chatbot"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7892
* To create a public link, set `share=True` in `launch()`.


c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 2026\deploying-ai\deploying-ai-env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



---
HumanMessage
content='Lover by Taylor Swift lyrics' additional_kwargs={} response_metadata={}

---
AIMessage
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 199, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8f3b023c79', 'id': 'chatcmpl-E1guVvqiGtSbjFRDRTu6MIc4PDukK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f62fa-1b6d-7961-83a4-187b410f3601-0' tool_calls=[{'name': 'get_lyrics', 'args': {'title': 'Lover', 'artist': 'Taylor Swift'}, 'id': 'call_bkevHIbIvDoVTgctkAr994zd', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 199, 'output_tokens': 21, 'total_tokens': 2

c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 2026\deploying-ai\deploying-ai-env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 2026\deploying-ai\deploying-ai-env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



---
HumanMessage
content='lyrics to yellow by coldplay' additional_kwargs={} response_metadata={}

---
AIMessage
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 199, 'total_tokens': 219, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8f3b023c79', 'id': 'chatcmpl-E1gvQnILP49TlB5KBUlWUdrh207NG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f62fa-fd4b-7c50-b2cb-bb21d82ecd00-0' tool_calls=[{'name': 'get_lyrics', 'args': {'title': 'Yellow', 'artist': 'Coldplay'}, 'id': 'call_ldVuI9eA6Vn9KStntOFqnypk', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 199, 'output_tokens': 20, 'total_tokens': 219,

c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 2026\deploying-ai\deploying-ai-env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 2026\deploying-ai\deploying-ai-env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



---
HumanMessage
content='best coldplay album' additional_kwargs={} response_metadata={}

---
AIMessage
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 197, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8f3b023c79', 'id': 'chatcmpl-E1gvfOp3JaKR4Z8DE0p4ba2oGq3LE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f62fb-3831-7842-a0c3-2a4f3632712e-0' tool_calls=[{'name': 'semantic_search', 'args': {'query': 'best Coldplay album'}, 'id': 'call_goNCqEX7eNzqi8GZsl3kzLDe', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 197, 'output_tokens': 17, 'total_tokens': 214, 'input_token

c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 2026\deploying-ai\deploying-ai-env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
